# Tema 1 - LLM & VLM basics
First of all, we need to setup and install the dependancies we will need for this project.

In [6]:
%pip install transformers accelerate torch torchvision pillow requests
%pip install -q qwen-vl-utils

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
### Tema 1A / Task 1A - LLM
> Incarcati un model LLM open-source mic (100M–1B parametri) pe laptop sau pe Colab si incercati prompturi de baza, generare si configurare folosind transformers.
> Pasi:
> 1. Alegeti si incarcati un model mai mic
> 2. Rulati un script simplu de generare text
> 3. Exemplificati cu diverse prompturi
> ---
> Load a small open-sourced LLM (100M-1B parameters) on your laptop or on Colab and try some base prompts, text generation, configurations using transformers.
> 1. Choose and Load a smaller model
> 2. Run a simple text-generation script.
> 3. Provide examples using various prompts.

In [7]:
###############################################################################################################
# Step 1 -> Loading 'distilgpt2' LLM which has ~82M parameters -> https://huggingface.co/distilbert/distilgpt2
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # alternativ: "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL_NAME}")
print(f"Parameters: {num_params/1e6:.1f}M")

Using device: cpu


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Parameters: 1100.0M


In [8]:
###############################################################################################################
# Step 2 -> Define a function to generate text using the loaded model
def generate(prompt, max_new_tokens=50, **gen_kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        max_length=None, # max_new_tokens takes precedence over max_length, so we set it to None to avoid console warnings.
        repetition_penalty=1.2, # Smaller models tends to repeat themselves, and be stuck in a loop. This is a simple way to reduce that.
        **gen_kwargs,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

prompt = "Once upon a time, in a small village,"
result = generate(prompt)
print(result)

Once upon a time, in a small village, there lived a young woman named Lily. She was kind and gentle, with long curly hair that she often wore in a ponytail. Her eyes were the color of emeralds, and her smile could light up any room. One


In [9]:
###############################################################################################################
# Step 3 -> Generate text using different decoding strategies
prompts = [
    "The best way to learn machine learning is",
    "In the future, artificial intelligence will",
    "def fibonacci(n):",
]

configs = {
    "Greedy (deterministic)": dict(do_sample=False),
    "Sampling temperature=0.7": dict(do_sample=True, temperature=0.7, top_k=50),
    "Sampling temperature=1.2, top_p=0.9": dict(do_sample=True, temperature=1.2, top_p=0.9),
}

for prompt in prompts:
    print("\n" + "=" * 80)
    print("PROMPT:", prompt)

    for config_name, kwargs in configs.items():
        text = generate(prompt, max_new_tokens=40, **kwargs)

        print(f"\n--- {config_name} ---")
        print(text[:1000])




PROMPT: The best way to learn machine learning is

--- Greedy (deterministic) ---
The best way to learn machine learning is by doing it. Start with small projects and work your way up, gradually increasing the complexity of the project as you gain more experience. 2. Read books: There are many great books on machine

--- Sampling temperature=0.7 ---
The best way to learn machine learning is by doing, and that's why we’re launching our free Machine Learning for Beginners course. This 5-part series covers everything from data preparation to model selection in Python using

--- Sampling temperature=1.2, top_p=0.9 ---
The best way to learn machine learning is through hands-on training material, videos , and online tutorials. Thanks for sharing your wonderful article. How can an experienced AI engineer contribute in the development of Deep Learning models using TensorFlow?

PROMPT: In the future, artificial intelligence will

--- Greedy (deterministic) ---
In the future, artificial intelli

---
### Tema 2B / Task 2B - LLM
> Incarcati un mic VLM HuggingFace
> 1. Prompt VLM pentru descrierea unei imagini sau recunoasterea unor obiecte dintr-o imagine.
> 2. Incercati un prompt care sa aiba ca rezultat identificarea / citirea unui text dintr-o imagine.
> ---
> Load a small HuggingFace's VLM
> 1. Create a VLM prompt for describing an image or recognizing objects in an image.
> 2. Try a prompt that results in identifying/reading text from an image.


In [1]:
###############################################################################################################
# Step 0 -> Loading 'Qwen2-VL-2B-Instruct' VLM -> https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch

VLM_MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"

vlm_model = Qwen2VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_NAME)
print("VLM model loaded successfully.")
print("VLM:", VLM_MODEL_NAME)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.


VLM model loaded successfully.
VLM: Qwen/Qwen2-VL-2B-Instruct


In [12]:
###
#Step 0.5 -> Helper Function to process the image and text input for the VLM
def ask_vlm(image_url_or_path, question, max_new_tokens=128):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_url_or_path},
                {"type": "text", "text": question},
            ],
        }
    ]

    text = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = vlm_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(vlm_model.device)

    generated_ids = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens)
    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = vlm_processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text[0]


In [13]:
###############################################################################################################
# Step 1 -> VLM describe image task 
image_url = "./DescribeIt.png"

answer = ask_vlm(image_url, "Describe this image in details and enumerate main objects you can see.")
print(answer)


The image depicts a highly detailed and intricate concept art of a mecha, specifically a Gundam, which is a popular series of mecha (mechanical robots) from the Japanese anime and manga genre. The mecha in the image is a large, humanoid robot with a highly detailed and complex structure. Here are the main objects and details visible in the image:

1. **Main Mecha**: The central focus of the image is a large, humanoid mecha with a highly detailed and intricate design. The mecha has a predominantly white color scheme with some dark accents, giving it a sleek and modern appearance. It features large, angular


In [15]:
###############################################################################################################
# Step 2 -> VLM read text from image task
image_with_text_url = "./ReadIt.jpg"

answer_ocr = ask_vlm(image_with_text_url, "What text appears in this image? Transcribe exactly the text you see.")
print(answer_ocr)

Bleach
Thousand-Year Blood War
